# Jaringan pabrik: penemuan mDNS, Sparkplug B, dan codec payload

Temukan perangkat tanpa server DNS, beri MQTT model state dengan Sparkplug B, dan baca payload biner yang Anda temui di jalan (Protobuf, MessagePack, TLV). Semuanya berjalan di proses ini.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.11.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.11.0-preview.1"
#r "nuget: IoTCom.Net.Serialization.Protobuf, 0.11.0-preview.1"
#r "nuget: IoTCom.Net.Serialization.MessagePack, 0.11.0-preview.1"

## Siapa saja di jaringan?
Browser bertanya ke 224.0.0.251:5353 untuk setiap tipe layanan, lalu untuk instance masing-masing. `MdnsSimulator` menjalankan segmen pabrik kecil di memori; `MdnsBrowser.Create()` tanpa opsi memakai jaringan sungguhan.

In [ ]:
using IoTCom.Net.Protocols.Mdns;

await using var plant = new MdnsSimulator();
await plant.StartAsync();
await using var browser = plant.Browser();
await browser.StartAsync();
foreach (var type in await browser.EnumerateTypesAsync(TimeSpan.FromMilliseconds(500)))
    foreach (var svc in await browser.BrowseAsync(type.Replace(".local", ""), TimeSpan.FromMilliseconds(300)))
        Console.WriteLine($"{svc.Type,-14} {svc.Instance,-24} {svc.Address}:{svc.Port}  {string.Join(" ", svc.Properties.Select(p => $"{p.Key}={p.Value}"))}");

## Sparkplug B: birth, data, dan death
Edge node mendaftarkan NDEATH sebagai will MQTT dan menerbitkan NBIRTH/DBIRTH dengan setiap metrik beserta alias. Host application menerjemahkan alias pada pesan DDATA berikutnya.

In [ ]:
using System.Net;
using System.Net.Sockets;
using IoTCom.Net.Adapters.Mqtt;
using IoTCom.Net.Protocols.Sparkplug;

var probe = new TcpListener(IPAddress.Loopback, 0);
probe.Start();
var mqttPort = ((IPEndPoint)probe.LocalEndpoint).Port;
probe.Stop();
var broker = MqttBroker.Create(mqttPort);
await broker.StartAsync();
var scada = SparkplugHost.Create(o => { o.HostId = "notebook"; o.Mqtt = m => m.UseBroker("127.0.0.1", mqttPort); });
await scada.StartAsync();
var edge = SparkplugEdgeNode.Create(o => { o.Group = "Plant"; o.EdgeNode = "Line1"; o.Mqtt = m => m.UseBroker("127.0.0.1", mqttPort); });
var bottling = new SparkplugLineSimulator(edge).Define();
await edge.StartAsync();
for (var i = 0; i < 3; i++) { await bottling.Step(); await Task.Delay(200); }
foreach (var v in scada.Views.OrderBy(v => v.Key))
    Console.WriteLine($"{v.Key,-20} {(v.Online ? "online " : "offline")} {string.Join(", ", v.Metrics.Values.Where(m => m.Name != "Node Control/Rebirth").Select(m => m.ToString()))}");

## Perintah dan kabel yang dicabut
Penulisan dikirim sebagai DCMD dan kembali sebagai DDATA. Saat koneksi putus, broker menerbitkan will NDEATH dan host menandai node beserta perangkatnya offline.

In [ ]:
await scada.WriteAsync("Plant", "Line1", "Filler", "Running", false);
await Task.Delay(300);
Console.WriteLine($"Filler running: {scada.Find("Plant", "Line1", "Filler")!.Metrics["Running"].Value}");
await edge.DropConnectionAsync();
await Task.Delay(500);
Console.WriteLine($"Line1 online: {scada.Find("Plant", "Line1")!.Online} · next bdSeq {edge.BdSeq}");
await scada.DisposeAsync();
await broker.DisposeAsync();

## Seperti apa payload Sparkplug
Ini Protobuf: `SparkplugPayload.Describe` memberi nama field, `ProtobufWire` membaca pesan apa pun tanpa skema.

In [ ]:
using IoTCom.Net.Serialization.Protobuf;

var payload = new SparkplugPayload { Timestamp = 1, Seq = 4, Metrics = [SparkplugMetric.Of("Level", SparkplugDataType.Double, 61.5, alias: 7)] }.Encode();
Console.WriteLine(Convert.ToHexString(payload));
foreach (var f in SparkplugPayload.Describe(payload)) Console.WriteLine($"{f.Name,-10} {f.Value}");
ProtobufWire.TryInspect(payload, out var wire);
foreach (var f in wire) Console.WriteLine($"#{f.Number} wire {f.WireType}: {f.Value}");

## MessagePack dan TLV
MessagePack adalah biner berbentuk JSON; BER-TLV adalah tata letak tag-length-value pada smart card dan EMV.

In [ ]:
using IoTCom.Net.Serialization.MessagePack;
using IoTCom.Net.Serialization.Tlv;

var packed = new MessagePackCodec<Dictionary<string, double>>(MessagePack.MessagePackSerializerOptions.Standard).Encode(new() { ["temp"] = 27.5, ["rh"] = 71 });
Console.WriteLine($"{Convert.ToHexString(packed)} → {MessagePackView.ToJson(packed)}");
var fci = Convert.FromHexString("6F148407A0000000031010A5095004564953419F3800");
void Show(IEnumerable<TlvItem> items, string indent = "") { foreach (var i in items) { Console.WriteLine($"{indent}{i.Tag:X} {(i.Children.Count > 0 ? "" : Convert.ToHexString(i.Value))}"); Show(i.Children, indent + "  "); } }
Show(BerTlv.Decode(fci));

## Lebih lanjut
`iotcom mdns browse`, `iotcom sparkplug watch --sim`, `iotcom payload protobuf <hex>`, demo *Jaringan pabrik* di Gallery, serta sampel MdnsDiscovery dan SparkplugEdgeNode. Lihat `docs/id/protocols/mdns.md`, `sparkplug.md`, dan `payload-codecs.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*